# 05 — Escala de turnos

Do dimensionamento à **escala**: quantas pessoas em cada turno, em cada dia, ao menor custo. Você vai gerar uma escala, ver as pausas de refeição escalonadas, conferir a cobertura, validar no simulador e testar um cenário "e se".

Leia antes o [guia de escala e cenários](../docs/08-guia-de-escala-e-cenarios.md). Este notebook **não precisa do modelo TimesFM**: usamos uma sexta-feira dos dados de exemplo.

## 1. O dimensionamento de uma sexta-feira

A escala parte da coluna **pessoas escaladas** (já com o shrinkage).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from callcenter_lab.wfm import StaffingConfig, build_plan

df = pd.read_csv(Path('..') / 'data' / 'samples' / 'chamadas_sinteticas.csv', parse_dates=['timestamp'])
day = df.set_index('timestamp')['calls_offered'].astype(float).loc['2024-05-17']  # uma sexta-feira
config = StaffingConfig(cost_per_hour=20)  # meta padrão de exemplo; custo de 20 por hora
plan = build_plan(day, config)
print(f"{int(day.sum())} chamadas | pico de {int(plan['agents'].max())} agentes atendendo | "
      f"pico de {int(plan['scheduled'].max())} pessoas escaladas")

## 2. Os turnos

Cada linha é um tipo de turno. Estes são **exemplos**: troque pelos da sua operação. Início e duração precisam ser múltiplos do intervalo (30 minutos).

In [ ]:
from callcenter_lab.wfm import example_templates
from callcenter_lab.wfm.scheduling import templates_to_table

templates = example_templates()
templates_to_table(templates)

## 3. Gerar a escala

`optimize_roster` resolve o problema de forma **exata** (programação inteira). Cobrir a necessidade é obrigatório na prática: faltar uma pessoa por um intervalo custa mais caro que um dia inteiro de trabalho dela.

In [ ]:
from callcenter_lab.wfm import RosterRules, optimize_roster

roster = optimize_roster(plan['scheduled'], templates, config.interval_minutes, RosterRules(default_cost_per_hour=20))
print('Solução:', roster.status)
print({k: round(v, 1) for k, v in roster.summary().items()})
roster.shifts

As **pausas de refeição** são escalonadas: a escala escolhe a hora da pausa de cada pessoa dentro da janela, para a cobertura não despencar na hora do almoço.

In [ ]:
roster.breaks

## 4. A cobertura contra a necessidade

In [ ]:
cov = roster.coverage
fig, ax = plt.subplots(figsize=(12, 4))
ax.step(cov.index, cov['covered'], where='post', label='Cobertura (no turno, sem pausa)')
ax.step(cov.index, cov['needed'], where='post', linestyle=':', color='tab:orange', label='Necessidade')
ax.set_ylabel('Pessoas')
ax.legend()
plt.show()

A sobra aparece onde a linha azul fica acima da laranja: turnos fixos de 8 horas não acompanham o formato do dia. Quanto de sobra?

In [ ]:
summary = roster.summary()
print(f"Sobra: {summary['surplus_person_hours']:.0f} horas | falta: {summary['shortfall_person_hours']:.0f} horas")

## 5. Validar no simulador

A escala entrega agentes diferentes da necessidade exata. Será que ainda cumpre a meta de 80% em 20 segundos?

In [ ]:
from callcenter_lab.wfm import evaluate_roster

check = evaluate_roster(roster, plan, config, n_replications=20, seed=1)
mean, half_width = check.overall['service_level_pct']
print(f'Nível de serviço simulado: {mean:.1f}% ± {half_width:.1f} (meta {config.service_level_pct:.0f}%)')

## 6. E se o volume vier no cenário alto?

A escala está montada. Quanto ela cobre se o volume for 30% maior que o esperado?

In [ ]:
from callcenter_lab.wfm import Scenario, run_scenarios

volumes = {'expected': day, 'high': day * 1.3}
scenarios = [Scenario('Base'), Scenario('Volume alto (+30%)', volume='high')]
result = run_scenarios(volumes, scenarios, config, roster=roster)
result[['scenario', 'peak_agents', 'agent_hours', 'cost', 'roster_shortfall_intervals', 'roster_shortfall_person_hours']].round(1)

Com volume alto, a escala fixa deixa intervalos descobertos. É a pergunta que o cenário responde antes de o dia chegar.

## 7. Turnos mais curtos reduzem a sobra?

Acrescentamos turnos de 4 horas nos horários de pico e comparamos.

In [ ]:
from callcenter_lab.wfm import ShiftTemplate

weekdays = (0, 1, 2, 3, 4)
flexible = [*templates,
            ShiftTemplate('Pico manhã 4h', '09:00', 4, weekdays),
            ShiftTemplate('Pico tarde 4h', '14:00', 4, weekdays)]
flex_roster = optimize_roster(plan['scheduled'], flexible, config.interval_minutes, RosterRules(default_cost_per_hour=20))
before, after = roster.summary(), flex_roster.summary()
print(f"Custo: {before['cost']:,.0f} -> {after['cost']:,.0f}")
print(f"Sobra: {before['surplus_person_hours']:.0f} h -> {after['surplus_person_hours']:.0f} h")

## 8. Levar para o Excel

`build_workbook` reúne tudo em uma planilha formatada (a mesma do botão da interface).

In [ ]:
from callcenter_lab.wfm import build_workbook

data = build_workbook(config=config, plan=plan, roster=roster, scenarios=result)
Path('dimensionamento_exemplo.xlsx').write_bytes(data)
print(f'{len(data):,} bytes salvos em dimensionamento_exemplo.xlsx')

## Para experimentar

- Troque o `max_days_per_week` para 6 em `RosterRules` e use uma semana inteira de dados (`df.loc['2024-05-13':'2024-05-19']`): quantas pessoas a equipe precisa?
- Dê um `max_on_break_pct=30` a um turno: as pausas ficam mais espalhadas?
- Limite as pessoas disponíveis com `RosterRules(max_headcount=...)` e veja a falta aparecer.